# 01 · Dataset Understanding - MEPS HC-243 (2022)

**Goal:** understand what the data *is* before any analysis: its files, shape, types, identifiers, variable groups, naming conventions, reserved codes, and potential targets.

No cleaning or modelling happens here. The raw file is loaded read-only.

## 1. Files

| File | Role |
|---|---|
| `data/raw/h243.xlsx` | **The dataset**: one sheet `H243`, loaded with pandas |
| `data/raw/h243doc.pdf` | **Documentation**: variable descriptions, reserved codes, weights. Not ML data. |

The xlsx takes ~2.5 minutes to parse. `load_raw()` caches an identical Parquet copy in `data/processed/` for fast reloads.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.data_loader import (
    RESERVED_CODES, SIGNED_INCOME_COLS, TARGET_COL, WEIGHT_COL, ID_COL,
    load_raw, expenditure_columns, variable_group_map,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [2]:
df = load_raw()
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]:,}")

Rows: 22,431   Columns: 1,420


## 2. First look

In [3]:
df.head()

,DUID,PID,DUPERSID,PANEL,DATAYEAR,FAMID31,FAMID42,FAMID53,FAMID22,FAMIDYR,CPSFAMID,FCSZ1231,FCRP1231,RULETR31,RULETR42,RULETR53,RULETR22,RUSIZE31,RUSIZE42,RUSIZE53,...,RXEXP22,RXSLF22,RXMCR22,RXMCD22,RXPRV22,RXVA22,RXTRI22,RXOFD22,RXSTL22,RXWCP22,RXOSR22,RXPTR22,RXOTH22,PERWT22F,FAMWT22F,FAMWT22C,SAQWT22F,DIABW22F,VARSTR,VARPSU
0,2460002,101,2460002101,24,2022,A,A,A,A,A,A,1,1,A,A,A,A,1,1,1,...,1602,109,1432,61,0,0,0,0,0,0,0,0,0,5728.309495,5232.211986,5232.211986,3994.687140,6034.636755,2082,1
1,2460006,101,2460006101,24,2022,A,A,A,A,A,A,1,1,A,A,A,A,2,2,2,...,1442,66,1376,0,0,0,0,0,0,0,0,0,0,15648.881461,16017.881691,16017.881691,0.000000,0.000000,2001,4
2,2460006,102,2460006102,24,2022,A,A,A,A,A,B,1,1,A,A,A,A,2,2,2,...,20,20,0,0,0,0,0,0,0,0,0,0,0,14123.720178,16017.881691,12580.731740,0.000000,0.000000,2001,4
3,2460010,101,2460010101,24,2022,A,A,A,A,A,A,1,1,A,A,A,A,1,1,1,...,6292,705,0,0,5288,0,0,40,259,0,0,5288,299,16982.054917,21905.758877,21905.758877,0.000000,0.000000,2038,3
4,2460018,101,2460018101,24,2022,A,A,A,A,A,A,1,1,A,A,A,A,1,1,1,...,252,242,0,0,10,0,0,0,0,0,0,10,0,10682.619947,11344.291012,11344.291012,17152.439412,0.000000,2041,1


In [4]:
df.info(verbose=False, memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 22431 entries, 0 to 22430
Columns: 1420 entries, DUID to VARPSU
dtypes: float64(19), int64(1390), str(11)
memory usage: 243.4 MB


In [5]:
print("dtype counts:")
print(df.dtypes.astype(str).value_counts().to_string())
print("\nNon-numeric columns:", df.select_dtypes(exclude="number").columns.tolist())

dtype counts:
int64      1390
float64      19
str          11

Non-numeric columns: ['FAMID31', 'FAMID42', 'FAMID53', 'FAMID22', 'FAMIDYR', 'CPSFAMID', 'RULETR31', 'RULETR42', 'RULETR53', 'RULETR22', 'HIEUIDX']


The 11 non-numeric columns are family/reporting-unit IDs and a health-insurance-eligibility-unit ID. They are identifiers, not features.

## 3. Integrity checks: NaNs, duplicates, identifiers

In [6]:
print("Total NaN cells:          ", int(df.isna().sum().sum()))
print("Duplicate rows:           ", int(df.duplicated().sum()))
print("Duplicate DUPERSID:       ", int(df[ID_COL].duplicated().sum()))
print("DUPERSID == DUID + PID:   ", (df[ID_COL].astype(str) == df["DUID"].astype(str) + df["PID"].astype(str).str.zfill(3)).all())
print("\nPersons per panel:")
print(df["PANEL"].value_counts().sort_index().to_string())

Total NaN cells:          

 0


Duplicate rows:            0
Duplicate DUPERSID:        0
DUPERSID == DUID + PID:    True

Persons per panel:
PANEL
24     5437
26     6843
27    10151


- **No NaN values anywhere.** Missing information is encoded with reserved negative codes (Section 6), so `isnull()` alone would wrongly report a complete dataset.
- `DUPERSID` uniquely identifies each person (household `DUID` + 3-digit person number `PID`).
- Three panels overlap in 2022. Panel 27 started in 2022, Panel 26 in 2021 and Panel 24 in 2019.

## 4. Variable groups

In [7]:
groups = pd.Series(variable_group_map(df.columns.tolist()), name="group")
summary = (groups.groupby(groups, sort=False).agg(n_columns="size",
           example_columns=lambda s: ", ".join(s.index[:6])))
display(summary)

,n_columns,example_columns
group,,
Survey admin / IDs,84,"DUID, PID, DUPERSID, PANEL, DATAYEAR, FAMID31"
Demographics,52,"AGE31X, AGE42X, AGE53X, AGE22X, AGELAST, DOBMM"
Priority conditions,59,"HIBPDX, HIBPAGED, BPMLDX, CHDDX, CHDAGED, ANGIDX"
Health status,196,"RTHLTH31, RTHLTH42, RTHLTH53, MNHLTH31, MNHLTH..."
Disability days,3,"DDNWRK22, OTHDYS22, OTHNDD22"
Access to care,30,"ACCELI42, HAVEUS42, PRACTP42, YNOUSC42_M18, PR..."
Employment,151,"EMPST31, EMPST42, EMPST53, RNDFLG31, MORJOB31,..."
Income & tax,51,"FILEDR22, WILFIL22, FLSTAT22, FILER22, JTINRU2..."
Health insurance,495,"TRIJA22X, TRIFE22X, TRIMA22X, TRIAP22X, TRIMY2..."


Health insurance has the most columns (495) because coverage is recorded **month by month** for each insurance type. Utilization & expenditure columns (292) break spending down by service type and by payer.

## 5. Variable naming conventions

- A trailing **`X`** means the variable was edited or imputed by AHRQ (`AGE22X`).
- **`31` / `42` / `53`** mean the same question asked in different interview rounds of 2022 (early / middle / late in the year).
- **`22`** marks a full-year 2022 or 31 December 2022 value.
- **Expenditure** names combine *service* + *payer* + `22`. For example `RXSLF22` = prescriptions paid out of pocket, and `TOTEXP22` = total payments from all sources.

Because of the round suffixes, many concepts appear 3–4 times:

In [8]:
import re
stems = pd.Series([re.sub(r"(31|42|53|22)(X)?$", "", c) for c in df.columns], index=df.columns)
families = stems.groupby(stems).apply(lambda s: list(s.index))
multi = families[families.str.len() >= 3]
print(f"{len(multi)} concepts appear in 3+ round versions, covering {multi.str.len().sum()} columns. Examples:")
for stem in ["AGE", "RTHLTH", "MNHLTH", "EMPST", "INSCOV", "REGION"]:
    if stem in families: print(f"  {stem:8} -> {families[stem]}")

100 concepts appear in 3+ round versions, covering 358 columns. Examples:
  AGE      -> ['AGE31X', 'AGE42X', 'AGE53X', 'AGE22X']
  RTHLTH   -> ['RTHLTH31', 'RTHLTH42', 'RTHLTH53']
  MNHLTH   -> ['MNHLTH31', 'MNHLTH42', 'MNHLTH53']
  EMPST    -> ['EMPST31', 'EMPST42', 'EMPST53']
  INSCOV   -> ['INSCOV22']
  REGION   -> ['REGION31', 'REGION42', 'REGION53', 'REGION22']


## 6. Reserved codes

The documentation (Section 2.2) defines the negative codes. Their frequency across all numeric cells:

In [9]:
num = df.select_dtypes("number").drop(columns=SIGNED_INCOME_COLS)
flat = pd.Series(num.to_numpy().ravel())
code_counts = flat[flat < 0].value_counts()
codes = pd.DataFrame({
    "meaning": [RESERVED_CODES.get(int(c), "?") for c in code_counts.index],
    "cells": code_counts.values,
    "pct_of_all_cells": (code_counts.values / flat.size * 100).round(2),
}, index=code_counts.index.astype(int))
display(codes)
print(f"Share of numeric cells holding a reserved code: {(flat < 0).mean():.1%}")

,meaning,cells,pct_of_all_cells
-1,Inapplicable,6646895,21.12
-2,Determined in previous round,353140,1.12
-15,Cannot be computed,73022,0.23
-8,Don't know,38644,0.12
-7,Refused,8574,0.03
-10,Hourly wage top-coded,541,0.00
-13,Initial wage imputed,446,0.00


Share of numeric cells holding a reserved code: 22.6%


- **`-1` Inapplicable** dominates. It is *structural*: the question was skipped by design (for example, adult-only questions for children). It is information, not random missingness.
- **`-7 / -8 / -15`** are true missing answers and are rare.

Some income columns contain **real negative values** (business or trust losses). They must not be treated as codes:

In [10]:
for c in SIGNED_INCOME_COLS:
    v = df[c]
    non_code = v[(v < 0) & ~v.isin(list(RESERVED_CODES))]
    print(f"{c:9}  negative non-code values: {len(non_code):4d}   e.g. {sorted(non_code.unique())[:4]}")

TTLP22X    negative non-code values:   19   e.g. [np.int64(-333392), np.int64(-317727), np.int64(-103120), np.int64(-75125)]
FAMINC22   negative non-code values:   17   e.g. [np.int64(-635454), np.int64(-333392), np.int64(-103120), np.int64(-75125)]
POVLEV22   negative non-code values:   17   e.g. [np.float64(-2189.77), np.float64(-1625.04), np.float64(-677.31), np.float64(-493.43)]
BUSNP22X   negative non-code values:   90   e.g. [np.int64(-333392), np.int64(-139000), np.int64(-90125), np.int64(-69509)]
SALEP22X   negative non-code values:   75   e.g. [np.int64(-215167), np.int64(-113548), np.int64(-105469), np.int64(-99770)]
TRSTP22X   negative non-code values:   82   e.g. [np.int64(-372452), np.int64(-336390), np.int64(-90555), np.int64(-49916)]


## 7. Column cardinality and constant columns

In [11]:
nun = df.nunique()
constant = nun[nun == 1].index.tolist()
print(f"Constant columns ({len(constant)}): {constant}")
bins = pd.cut(nun, [0, 1, 2, 10, 50, 1_000, np.inf],
              labels=["1 (constant)", "2", "3-10", "11-50", "51-1000", ">1000"])
print("\nColumns by number of distinct values:")
print(bins.value_counts().sort_index().to_string())

Constant columns (13): ['DATAYEAR', 'ENDRFY22', 'OPDOFD22', 'OPSOFD22', 'OPSSTL22', 'IPDOFD22', 'IPDSTL22', 'HHAOFD22', 'HHAWCP22', 'HHNOFD22', 'HHNSTL22', 'HHNWCP22', 'VISWCP22']

Columns by number of distinct values:
1 (constant)     13
2                35
3-10            965
11-50           135
51-1000         192
>1000            80


Most columns are low-cardinality coded categories (yes/no plus reserved codes). High-cardinality columns are counts, ages, incomes, dollar amounts and weights. Constant columns carry no information and will be dropped.

## 8. Identifiers and survey weights

In [12]:
print(f"Persons with PERWT22F = 0 (no person-level weight): {(df[WEIGHT_COL] == 0).sum():,}")
print(f"Persons with PERWT22F > 0 (analysis population):    {(df[WEIGHT_COL] > 0).sum():,}")
print(f"Sum of weights (U.S. civilian non-institutionalized population, 2022): {df[WEIGHT_COL].sum():,.0f}")

Persons with PERWT22F = 0 (no person-level weight): 684
Persons with PERWT22F > 0 (analysis population):    21,747
Sum of weights (U.S. civilian non-institutionalized population, 2022): 333,053,243


| Role | Columns | Use in ML |
|---|---|---|
| Identifiers | `DUID`, `PID`, `DUPERSID`, `FAMID*`, `CPSFAMID`, `RULETR*`, `HIEUIDX`, `PANEL` | Never features |
| Survey weights & design | `PERWT22F`, `FAMWT22F`, `SAQWT22F`, `DIABW22F`, `VARSTR`, `VARPSU`… | Not features. `PERWT22F > 0` defines the analysis population |

## 9. Potential targets

In [13]:
exp_cols = expenditure_columns(df)
components = ["OBVEXP22", "OPTEXP22", "ERTEXP22", "IPTEXP22", "RXEXP22", "DVTEXP22",
              "HHAEXP22", "HHNEXP22", "VISEXP22", "OTHEXP22"]
match = (df[components].sum(axis=1) - df[TARGET_COL]).abs() < 2
print(f"TOTEXP22 equals the sum of the 10 service-level expenditure columns for {match.mean():.1%} of rows")
print(f"Payment / source-of-payment / charge columns derived from the same events: {len(exp_cols)}")
display(df[TARGET_COL].describe().to_frame().T.round(0))

TOTEXP22 equals the sum of the 10 service-level expenditure columns for 100.0% of rows
Payment / source-of-payment / charge columns derived from the same events: 279


,count,mean,std,min,25%,50%,75%,max
TOTEXP22,22431.0,7639.0,22994.0,0.0,260.0,1605.0,6409.0,1338276.0


| Candidate | Type | ML problem |
|---|---|---|
| `TOTEXP22` (total 2022 payments) | continuous, ≥ 0 | **Regression** |
| `TOTEXP22` ≥ threshold (high cost) | binary | **Binary classification** (main project problem, threshold set in EDA) |
| none | — | **Clustering / PCA** on person characteristics |

**Leakage warning:** because `TOTEXP22` is exactly the sum of its components, all 279 expenditure, payment and charge columns must be excluded from the predictors.

## Summary

- 22,431 persons × 1,420 variables, one row per person, no NaNs and no duplicates.
- Missing information uses reserved codes, mostly structural `-1` (skip patterns). Incomes can be genuinely negative.
- Many variables repeat across rounds (`31/42/53/22`), so the real number of distinct concepts is much smaller than 1,420.
- The analysis population is the 21,747 persons with `PERWT22F > 0`.
- `TOTEXP22` is the target. Its component columns are leakage.

Next: `02_eda.ipynb`.